# Arize Phoenix Tracing & Observability Playground
This notebook demonstrates how to instrument python workflows to log OTEL traces, construct span hierarchies, and query them inside the Arize Phoenix UI using the configured Otlp endpoints.

In [1]:
import sys
from pathlib import Path

# Locate and append project root
def get_project_root() -> Path:
    current = Path.cwd()
    for _ in range(5):
        if (current / "pyproject.toml").exists():
            return current
        current = current.parent
    return Path.cwd()

sys.path.append(str(get_project_root().resolve()))

In [2]:
from opentelemetry import trace
from opentelemetry.sdk.trace import TracerProvider
from opentelemetry.sdk.trace.export import SimpleSpanProcessor
from opentelemetry.exporter.otlp.proto.http.trace_exporter import OTLPSpanExporter
from ragforge.config import PHOENIX_COLLECTOR_URL
import time

print(f"Arize Phoenix OTEL Collector Endpoint: {PHOENIX_COLLECTOR_URL}")

Arize Phoenix OTEL Collector Endpoint: http://localhost:6006/v1/traces


## 1. Setup OpenTelemetry SDK Tracer Provider

In [3]:
# Initialize Tracer provider linking OTLPSpanExporter to Arize Phoenix
provider = TracerProvider()
processor = SimpleSpanProcessor(OTLPSpanExporter(endpoint=PHOENIX_COLLECTOR_URL))
provider.add_span_processor(processor)
trace.set_tracer_provider(provider)

tracer = trace.get_tracer("notebook-observability-tracer")
print("OTEL Tracer provider bound successfully!")

OTEL Tracer provider bound successfully!


## 2. Simulate Instrumented Workflow Execution Spans

In [4]:
# Simulate a nested trace workflow
with tracer.start_as_current_span("IngestionWorkflow") as root_span:
    root_span.set_attribute("directory_path", "notebooks/test_doc.txt")
    
    # Step 1: Scan
    with tracer.start_as_current_span("ScanDirectory") as step1:
        time.sleep(0.1)
        step1.set_attribute("files_found", ["test_doc.txt"])
        
    # Step 2: Parse & Chunk
    with tracer.start_as_current_span("ParseAndChunk") as step2:
        time.sleep(0.3)
        step2.set_attribute("chunks_generated", 15)
        
    # Step 3: Embed & Index
    with tracer.start_as_current_span("EmbedAndIndex") as step3:
        time.sleep(0.4)
        step3.set_attribute("collection_name", "ragforge-collection")
        step3.set_attribute("status", "success")

print("Workflow spans submitted! Navigate to http://localhost:6006 to inspect them.")

Workflow spans submitted! Navigate to http://localhost:6006 to inspect them.
